In [ ]:
import pandas as pd

df = pd.read_csv("../static/data/data_sensors.csv")
df_labeled = df[~df['Label'].isna()]
df_unlabeled = df[df['Label'].isna()]

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

# ── 1. All sensor distributions on one plot ──────────────────────────────────
sensor_cols = [c for c in df.columns if c.startswith('Sensor')]
n = len(sensor_cols)
colors_sensors = cm.tab20(np.linspace(0, 1, n))

fig, (ax_dist, ax_time) = plt.subplots(1, 2, figsize=(16, 6))

for i, col in enumerate(sensor_cols):
    ax_dist.hist(df[col].dropna(), bins=30, alpha=0.4, color=colors_sensors[i], label=col)
    ax_time.plot(df.index, df[col], linewidth=0.7, alpha=0.6, color=colors_sensors[i], label=col)

ax_dist.set_title("All sensors — distribution")
ax_dist.set_xlabel("value")
ax_dist.set_ylabel("count")
ax_dist.legend(fontsize=7, ncol=2)

ax_time.set_title("All sensors — readings over time")
ax_time.set_xlabel("index (time)")
ax_time.set_ylabel("value")
ax_time.legend(fontsize=7, ncol=2)

plt.suptitle("Sensor overview", fontsize=14)
plt.tight_layout()
plt.show()

# Iterate through each sensor individually
for i, col in enumerate(sensor_cols):
    # Create a new figure for each sensor
    fig, (ax_dist, ax_time) = plt.subplots(1, 2, figsize=(16, 5))

    # 1. Histogram (Distribution)
    ax_dist.hist(df[col].dropna(), bins=30, alpha=0.7,
                 color=colors_sensors[i % len(colors_sensors)], edgecolor='black')
    ax_dist.set_title(f"{col} — Distribution")
    ax_dist.set_xlabel("Value")
    ax_dist.set_ylabel("Count")

    # 2. Time Series (Readings over time)
    ax_time.plot(df.index, df[col], linewidth=1,
                 color=colors_sensors[i % len(colors_sensors)])
    ax_time.set_title(f"{col} — Readings over Time")
    ax_time.set_xlabel("Index (Time)")
    ax_time.set_ylabel("Value")

    # Add a main title for this specific sensor's figure
    plt.suptitle(f"Analysis for {col}", fontsize=14)

    plt.tight_layout(rect=[0, 0.03, 1, 0.95]) # Adjust layout to make room for suptitle
    plt.show() # Or plt.savefig(f"plot_{col}.png")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

labels = df_labeled['Label'].values
unique_labels = sorted(set(labels))
counts = [np.sum(labels == l) for l in unique_labels]
colors = cm.tab10(np.linspace(0, 0.4, len(unique_labels)))

# ── Pie chart ────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 7))
wedges, texts, autotexts = ax.pie(
    counts,
    labels=[f"Class {int(l)}" for l in unique_labels],
    colors=colors,
    autopct=lambda pct: f"{pct:.1f}%\n({int(round(pct * sum(counts) / 100))})",
    startangle=140,
    pctdistance=0.75,
)
for at in autotexts:
    at.set_fontsize(9)
ax.set_title("Label distribution (labeled data)", fontsize=13)
plt.tight_layout()
plt.show()

# ── Box + violin: labelled (by class) vs unlabeled ────────────────────────────
focus_sensors = ['Sensor 2', 'Sensor 9', 'Sensor 13']

def _draw_violin_box(ax, data_groups, positions, group_colors, x_labels):
    parts = ax.violinplot(data_groups, positions=positions, showextrema=False, widths=0.6)
    for i, pc in enumerate(parts['bodies']):
        pc.set_facecolor(group_colors[i])
        pc.set_alpha(0.45)
    bp = ax.boxplot(data_groups, positions=positions, widths=0.25, patch_artist=True,
                    medianprops=dict(color='black', linewidth=2),
                    whiskerprops=dict(linewidth=1.2),
                    capprops=dict(linewidth=1.2),
                    flierprops=dict(marker='o', markersize=3, alpha=0.4))
    for i, patch in enumerate(bp['boxes']):
        patch.set_facecolor(group_colors[i])
        patch.set_alpha(0.75)
    ax.set_xticks(positions)
    ax.set_xticklabels(x_labels)

for sensor in focus_sensors:
    fig, (ax_lab, ax_unlab) = plt.subplots(1, 2, figsize=(14, 5))
    fig.suptitle(f"{sensor} — box + violin", fontsize=13)

    # Labeled — one group per class
    data_labeled = [df_labeled.loc[labels == l, sensor].values for l in unique_labels]
    x_pos = np.arange(len(unique_labels))
    _draw_violin_box(ax_lab, data_labeled, x_pos, colors,
                     [f"Class {int(l)}" for l in unique_labels])
    ax_lab.set_xlabel("Label")
    ax_lab.set_ylabel("Value")
    ax_lab.set_title("Labeled")

    # Unlabeled — single distribution
    data_unlabeled = [df_unlabeled[sensor].dropna().values]
    _draw_violin_box(ax_unlab, data_unlabeled, [0], ['steelblue'], ['unlabeled'])
    ax_unlab.set_xlabel("")
    ax_unlab.set_ylabel("Value")
    ax_unlab.set_title("Unlabeled")

    plt.tight_layout()
    plt.show()


In [ ]:
## Review how the data looks like for the three sensors when projected on them
from itertools import combinations

focus_sensors = ['Sensor 2', 'Sensor 9', 'Sensor 13']
pairs = list(combinations(focus_sensors, 2))

labels = df_labeled['Label'].values
unique_labels = sorted(set(labels))
colors = cm.tab10(np.linspace(0, 0.4, len(unique_labels)))
label_color_map = {l: c for l, c in zip(unique_labels, colors)}

fig, axes = plt.subplots(1, len(pairs), figsize=(6 * len(pairs), 5))
fig.suptitle("Pairwise sensor scatter — labeled (by class) + unlabeled (grey)", fontsize=13)

for ax, (sx, sy) in zip(axes, pairs):
    # unlabeled underneath
    ax.scatter(df_unlabeled[sx], df_unlabeled[sy],
               color='lightgrey', edgecolor='none', s=15, alpha=0.5, label='Unlabeled', zorder=1)
    # labeled on top, colored by class
    for l in unique_labels:
        mask = labels == l
        ax.scatter(df_labeled.loc[mask, sx], df_labeled.loc[mask, sy],
                   color=label_color_map[l], edgecolor='black', linewidths=0.3,
                   s=25, alpha=0.85, label=f'Class {int(l)}', zorder=2)
    ax.set_xlabel(sx)
    ax.set_ylabel(sy)
    ax.set_title(f"{sx} vs {sy}")

axes[0].legend(fontsize=8, markerscale=1.2)
plt.tight_layout()
plt.show()

In [ ]:
# Many other sensors look like white noise? - can I get rid of them?
from statsmodels.tsa.stattools import ccf
a = df["Sensor 0"].values
b = df["Sensor 1"].values
out = ccf(a, b, nlags=50, alpha=0.05)

from matplotlib import rcParams
from statsmodels.graphics.tsaplots import plot_acf
rcParams["figure.figsize"] = 9, 4
# ACF function up to 50 lags
fig = plot_acf(a, lags=50)

In [ ]:
# Between the three sensors that seems to be informative - are any of them cross-correlated?
from itertools import combinations
from scipy.stats import pearsonr
from scipy.signal import coherence
from statsmodels.tsa.stattools import ccf

focus_sensors = ['Sensor 2', 'Sensor 9', 'Sensor 13']
nlags = 50

for sx, sy in combinations(focus_sensors, 2):
    a, b = df[sx].values, df[sy].values
    r, p = pearsonr(a, b)

    corr, conf = ccf(a, b, nlags=nlags, alpha=0.05)
    lags = np.arange(len(corr))
    lower = conf[:, 0] - corr
    upper = conf[:, 1] - corr

    freqs, Cxy = coherence(a, b, fs=100, nperseg=128)

    fig, (ax_ccf, ax_coh) = plt.subplots(1, 2, figsize=(14, 4))

    ax_ccf.vlines(lags, 0, corr, linewidth=1.2)
    ax_ccf.fill_between(lags, lower, upper, alpha=0.2, label="95% CI")
    ax_ccf.axhline(0, color="black", linewidth=0.8)
    ax_ccf.set(xlabel="Lag", ylabel="CCF",
               title=f"CCF: {sx} → {sy}   (Pearson r = {r:.3f}, p = {p:.2e})")
    ax_ccf.legend(fontsize=8)

    ax_coh.plot(freqs, Cxy, linewidth=1.2)
    ax_coh.set(xlabel="Frequency", ylabel="Coherence [0–1]",
               title=f"Coherence: {sx} vs {sy}", ylim=(0, 1))
    ax_coh.axhline(0.5, color="grey", linewidth=0.8, linestyle="--", label="0.5 threshold")
    ax_coh.legend(fontsize=8)

    plt.tight_layout()
    plt.show()


In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE

X = df_labeled[sensor_cols].values
labels = df_labeled['Label'].values
unique_labels = sorted(set(labels))
colors = cm.tab10(np.linspace(0, 0.4, len(unique_labels)))
label_color_map = {l: c for l, c in zip(unique_labels, colors)}

angles = [(20, 0), (20, 60), (20, 120), (45, 180), (70, 240), (10, 300)]

# ── 2. Linear PCA → 3D  ───────────────────────────────────────────
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X)
explained = pca.explained_variance_ratio_ * 100

fig = plt.figure(figsize=(18, 10))
fig.suptitle("PCA — 3D projection (labeled data)", fontsize=14)
for idx, (elev, azim) in enumerate(angles):
    ax = fig.add_subplot(2, 3, idx + 1, projection='3d')
    for l in unique_labels:
        mask = labels == l
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1], X_pca[mask, 2],
                   label=f"Class {int(l)}", s=30, alpha=0.85, color=label_color_map[l])
    ax.view_init(elev=elev, azim=azim)
    ax.set_xlabel(f"PC1 ({explained[0]:.1f}%)", fontsize=7)
    ax.set_ylabel(f"PC2 ({explained[1]:.1f}%)", fontsize=7)
    ax.set_zlabel(f"PC3 ({explained[2]:.1f}%)", fontsize=7)
    ax.set_title(f"elev={elev}°, azim={azim}°", fontsize=9)
    if idx == 0:
        ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

# ── 3. Non-linear t-SNE → 3D  ─────────────────────────────────────
tsne = TSNE(n_components=3, random_state=42, perplexity=min(30, len(X) - 1))
X_tsne = tsne.fit_transform(X)

fig = plt.figure(figsize=(18, 10))
fig.suptitle("t-SNE — 3D projection (labeled data)", fontsize=14)
for idx, (elev, azim) in enumerate(angles):
    ax = fig.add_subplot(2, 3, idx + 1, projection='3d')
    for l in unique_labels:
        mask = labels == l
        ax.scatter(X_tsne[mask, 0], X_tsne[mask, 1], X_tsne[mask, 2],
                   label=f"Class {int(l)}", s=30, alpha=0.85, color=label_color_map[l])
    ax.view_init(elev=elev, azim=azim)
    ax.set_xlabel("t-SNE 1", fontsize=7)
    ax.set_ylabel("t-SNE 2", fontsize=7)
    ax.set_zlabel("t-SNE 3", fontsize=7)
    ax.set_title(f"elev={elev}°, azim={azim}°", fontsize=9)
    if idx == 0:
        ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

############
# Unlabeled
############
# ── 2. Linear PCA → 3D ──────────────────────────────────────────────────────
X = df_unlabeled[sensor_cols].values

pca = PCA(n_components=3)
X_pca = pca.fit_transform(X)

fig = plt.figure(figsize=(9, 7))
ax = fig.add_subplot(111, projection='3d')
ax.scatter(X_pca[:, 0], X_pca[:, 1], X_pca[:, 2], s=50, alpha=0.85, color='steelblue')
explained = pca.explained_variance_ratio_ * 100
ax.set_xlabel(f"PC1 ({explained[0]:.1f}%)")
ax.set_ylabel(f"PC2 ({explained[1]:.1f}%)")
ax.set_zlabel(f"PC3 ({explained[2]:.1f}%)")
ax.set_title("PCA — 3D projection (unlabeled data)")
plt.tight_layout()
plt.show()


# ── 3. Non-linear t-SNE → 3D  ─────────────────────────────────────
# T-distributed Stochastic Neighbour Embedding.
# t-SNE is data hungry needs 30+ observations but estimates lower dimensional space via "local neighbourhoods"
# can play with local neighbourhoods via perxplexity param

# tsne = TSNE(n_components=3, random_state=42, perplexity=min(30, len(X) - 1))
# X_tsne = tsne.fit_transform(X)
#
# fig = plt.figure(figsize=(9, 7))
# ax = fig.add_subplot(111, projection='3d')
# ax.scatter(X_tsne[:, 0], X_tsne[:, 1], X_tsne[:, 2], s=50, alpha=0.85, color='steelblue')
# ax.set_xlabel("t-SNE 1")
# ax.set_ylabel("t-SNE 2")
# ax.set_zlabel("t-SNE 3")
# ax.set_title("t-SNE — 3D projection (unlabeled data)")
# plt.tight_layout()
# plt.show()

# ── 4.UMAP LOL? ────────────────────────────────────────────────
# import umap
#
# X = df_labeled[sensor_cols].values
# labels = df_labeled['Label'].values
# unique_labels = sorted(set(labels))
# colors = cm.tab10(np.linspace(0, 0.4, len(unique_labels)))
# label_color_map = {l: c for l, c in zip(unique_labels, colors)}
#
# angles = [(20, 0), (20, 60), (20, 120), (45, 180), (70, 240), (10, 300)]
#
# reducer = umap.UMAP(n_components=3, random_state=42)
# X_umap = reducer.fit_transform(X)
#
# fig = plt.figure(figsize=(18, 10))
# fig.suptitle("UMAP — 3D projection (labeled data)", fontsize=14)
# for idx, (elev, azim) in enumerate(angles):
#     ax = fig.add_subplot(2, 3, idx + 1, projection='3d')
#     for l in unique_labels:
#         mask = labels == l
#         ax.scatter(X_umap[mask, 0], X_umap[mask, 1], X_umap[mask, 2],
#                    label=f"Class {int(l)}", s=30, alpha=0.85, color=label_color_map[l])
#     ax.view_init(elev=elev, azim=azim)
#     ax.set_xlabel("UMAP 1", fontsize=7)
#     ax.set_ylabel("UMAP 2", fontsize=7)
#     ax.set_zlabel("UMAP 3", fontsize=7)
#     ax.set_title(f"elev={elev}°, azim={azim}°", fontsize=9)
#     if idx == 0:
#         ax.legend(fontsize=7)
# plt.tight_layout()
# plt.show()

In [ ]:
## Plot only specif sensor with non-uniform distribution

X = df_labeled[['Sensor 2', 'Sensor 9', 'Sensor 13']].values
labels = df_labeled['Label'].values
unique_labels = sorted(set(labels))
colors = cm.tab10(np.linspace(0, 0.4, len(unique_labels)))
label_color_map = {l: c for l, c in zip(unique_labels, colors)}

angles = [(20, 0), (20, 60), (20, 120), (45, 180), (70, 240), (10, 300)]

fig = plt.figure(figsize=(18, 10))
fig.suptitle("Sensor 2 / 9 / 13 — 3D scatter (labeled data)", fontsize=14)
for idx, (elev, azim) in enumerate(angles):
    ax = fig.add_subplot(2, 3, idx + 1, projection='3d')
    for l in unique_labels:
        mask = labels == l
        ax.scatter(X[mask, 0], X[mask, 1], X[mask, 2],
                   label=f"Class {int(l)}", s=30, alpha=0.85, color=label_color_map[l])
    ax.view_init(elev=elev, azim=azim)
    ax.set_xlabel("Sensor 2", fontsize=7)
    ax.set_ylabel("Sensor 9", fontsize=7)
    ax.set_zlabel("Sensor 13", fontsize=7)
    ax.set_title(f"elev={elev}°, azim={azim}°", fontsize=9)
    if idx == 0:
        ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
import numpy as np
from sklearn.preprocessing import PolynomialFeatures

# Raw input features
X = df_labeled[['Sensor 2', 'Sensor 9', 'Sensor 13']].values
labels = df_labeled['Label'].values

# Combine features
poly = PolynomialFeatures(8)
X = poly.fit_transform(X)
labels = df_labeled['Label'].values
unique_labels = sorted(set(labels))
colors = cm.tab10(np.linspace(0, 0.4, len(unique_labels)))
label_color_map = {l: c for l, c in zip(unique_labels, colors)}


angles = [(20, 0), (20, 60), (20, 120), (45, 180), (70, 240), (10, 300)]

# ── 2. Linear PCA → 3D  ───────────────────────────────────────────
pca = PCA(n_components=3)
X_pca = pca.fit_transform(X)
explained = pca.explained_variance_ratio_ * 100

fig = plt.figure(figsize=(18, 10))
fig.suptitle("PCA — 3D projection (labeled data)", fontsize=14)
for idx, (elev, azim) in enumerate(angles):
    ax = fig.add_subplot(2, 3, idx + 1, projection='3d')
    for l in unique_labels:
        mask = labels == l
        ax.scatter(X_pca[mask, 0], X_pca[mask, 1], X_pca[mask, 2],
                   label=f"Class {int(l)}", s=30, alpha=0.85, color=label_color_map[l])
    ax.view_init(elev=elev, azim=azim)
    ax.set_xlabel(f"PC1 ({explained[0]:.1f}%)", fontsize=7)
    ax.set_ylabel(f"PC2 ({explained[1]:.1f}%)", fontsize=7)
    ax.set_zlabel(f"PC3 ({explained[2]:.1f}%)", fontsize=7)
    ax.set_title(f"elev={elev}°, azim={azim}°", fontsize=9)
    if idx == 0:
        ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

# ── 3. Non-linear t-SNE → 3D  ─────────────────────────────────────
# T-distributed Stochastic Neighbour Embedding.
# t-SNE is data hungry needs 30+ observations but estimates lower dimensional space via "local neighbourhoods"

In [ ]:
## Plot only specif sensor and select two labels

df_labeled_1_2  = df_labeled[df_labeled.Label.isin([1,2])]
X = df_labeled_1_2[['Sensor 2', 'Sensor 9', 'Sensor 13']].values
labels = df_labeled_1_2['Label'].values

unique_labels = sorted(set(labels))
label_color_map = {1: 'steelblue', 2: 'mediumseagreen'}

angles = [(20, 0), (20, 60), (20, 120), (45, 180), (70, 240), (10, 300)]

fig = plt.figure(figsize=(18, 10))
fig.suptitle("Sensor 2 / 9 / 13 — 3D scatter (labeled data)", fontsize=14)
for idx, (elev, azim) in enumerate(angles):
    ax = fig.add_subplot(2, 3, idx + 1, projection='3d')
    for l in unique_labels:
        mask = labels == l
        ax.scatter(X[mask, 0], X[mask, 1], X[mask, 2],
                   label=f"Class {int(l)}", s=30, alpha=0.85, color=label_color_map[l])
    ax.view_init(elev=elev, azim=azim)
    ax.set_xlabel("Sensor 2", fontsize=7)
    ax.set_ylabel("Sensor 9", fontsize=7)
    ax.set_zlabel("Sensor 13", fontsize=7)
    ax.set_title(f"elev={elev}°, azim={azim}°", fontsize=9)
    if idx == 0:
        ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
## Plot only sensor readings from 2 and 9 and the radios dimension

X = df_labeled[['Sensor 2', 'Sensor 9', 'Sensor 13']].values
X = np.concat([X, np.sqrt(np.sum(X * X, axis=1)).reshape(-1,1)], axis=1)
X = X[:, [0,1,3]]

labels = df_labeled['Label'].values
unique_labels = sorted(set(labels))
colors = cm.tab10(np.linspace(0, 0.4, len(unique_labels)))
label_color_map = {l: c for l, c in zip(unique_labels, colors)}

angles = [(20, 0), (20, 60), (20, 120), (45, 180), (70, 240), (10, 300)]

fig = plt.figure(figsize=(18, 10))
fig.suptitle("Sensor 2 / 9 / R — 3D scatter (labeled data)", fontsize=14)
for idx, (elev, azim) in enumerate(angles):
    ax = fig.add_subplot(2, 3, idx + 1, projection='3d')
    for l in unique_labels:
        mask = labels == l
        ax.scatter(X[mask, 0], X[mask, 1], X[mask, 2],
                   label=f"Class {int(l)}", s=30, alpha=0.85, color=label_color_map[l])
    ax.view_init(elev=elev, azim=azim)
    ax.set_xlabel("Sensor 2", fontsize=7)
    ax.set_ylabel("Sensor 9", fontsize=7)
    ax.set_zlabel("Sensor R", fontsize=7)
    ax.set_title(f"elev={elev}°, azim={azim}°", fontsize=9)
    if idx == 0:
        ax.legend(fontsize=7)
plt.tight_layout()
plt.show()

In [ ]:
## Sensor 9 vs R — 2D scatter

X = df_labeled[['Sensor 2', 'Sensor 9', 'Sensor 13']].values
X = np.concat([X, np.sqrt(np.sum(X * X, axis=1)).reshape(-1, 1)], axis=1)
X = X[:, [1, 3]]  # Sensor 9, R

labels = df_labeled['Label'].values
unique_labels = sorted(set(labels))
colors = cm.tab10(np.linspace(0, 0.4, len(unique_labels)))
label_color_map = {l: c for l, c in zip(unique_labels, colors)}

fig, ax = plt.subplots(figsize=(8, 6))
for l in unique_labels:
    mask = labels == l
    ax.scatter(X[mask, 0], X[mask, 1],
               label=f"Class {int(l)}", s=20, alpha=0.7, color=label_color_map[l])
ax.set_xlabel("Sensor 9")
ax.set_ylabel("R")
ax.set_title("Sensor 9 vs R — 2D scatter (labeled data)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
## Sensor 2 vs R — 2D scatter

X = df_labeled[['Sensor 2', 'Sensor 9', 'Sensor 13']].values
X = np.concat([X, np.sqrt(np.sum(X * X, axis=1)).reshape(-1, 1)], axis=1)
X = X[:, [0, 3]]  # Sensor 2, R

labels = df_labeled['Label'].values
unique_labels = sorted(set(labels))
colors = cm.tab10(np.linspace(0, 0.4, len(unique_labels)))
label_color_map = {l: c for l, c in zip(unique_labels, colors)}

fig, ax = plt.subplots(figsize=(8, 6))
for l in unique_labels:
    mask = labels == l
    ax.scatter(X[mask, 0], X[mask, 1],
               label=f"Class {int(l)}", s=20, alpha=0.7, color=label_color_map[l])
ax.set_xlabel("Sensor 2")
ax.set_ylabel("R")
ax.set_title("Sensor 2 vs R — 2D scatter (labeled data)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
## See how the unlabeled data points look on our new axis
X = df_unlabeled[['Sensor 2', 'Sensor 9', 'Sensor 13']].values
X = np.concat([X, np.sqrt(np.sum(X * X, axis=1)).reshape(-1, 1)], axis=1)
X = X[:, [1, 3]]  # Sensor 9, R

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(X[:, 0], X[:, 1], label=f"Unlabeled", s=20, alpha=0.5)
ax.set_xlabel("Sensor 9")
ax.set_ylabel("R")
ax.set_title("Sensor 9 vs R — 2D scatter (unlabeled data)")
ax.legend(fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
## Inspect classification and label the rest of observations based on the feature engineering performed
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np

from sklearn.inspection import DecisionBoundaryDisplay
from sklearn.semi_supervised import LabelSpreading, SelfTrainingClassifier
from sklearn.svm import SVC
from matplotlib.colors import ListedColormap

X = df[['Sensor 2', 'Sensor 9', 'Sensor 13']].values
X = np.concat([X, np.sqrt(np.sum(X * X, axis=1)).reshape(-1, 1)], axis=1)
X = X[:, [1, 3]]  # Sensor 9, R
df["label_label_spreading"] = df["Label"].fillna(-1)
y = df["label_label_spreading"].values

labels = df['label_label_spreading'].values
unique_labels = sorted(set(labels))

# A bit of handcrafting so that the colours match previous ones
unique_labels_without_unlabeled = [i for i in unique_labels if i!=-1]
colors = cm.tab10(np.linspace(0, 0.4, len(unique_labels_without_unlabeled)))
label_color_map = {l: c for l, c in zip(unique_labels_without_unlabeled, colors)}
label_color_map[-1] = (1, 1, 1)
# For plotting regions later
region_colors = [label_color_map[l] for l in sorted(unique_labels) if l != -1]
cmap = ListedColormap(region_colors)

# Default kernel is RBF for svc (https://scikit-learn.org/stable/modules/svm.html#svm-kernels)
#  - iterate over different hyperparam values
hyper_params = [({"gamma": 10}, {"gamma": 0.2}), ({"gamma": 20}, {"gamma": 0.5}), ({"gamma": 25}, {"gamma": 0.7}), ({"gamma": 70}, {"gamma": 1.2})]
for hyper_param_ls,hyper_param_svc in hyper_params:
    ls100 = (LabelSpreading(**hyper_param_ls).fit(X, y), y, "LabelSpreading")
    base_classifier = SVC(probability=True, random_state=42, **hyper_param_svc)
    st10 = (
        SelfTrainingClassifier(base_classifier).fit(X, y),
        y,
        "SVC",
    )
    classifiers = (ls100, st10)

    fig, axes = plt.subplots(nrows=1, ncols=2, sharex="col", sharey="row", figsize=(14, 6))

    handles = [
        mpatches.Patch(facecolor=label_color_map[i], edgecolor="black", label=i)
        for i in unique_labels
    ]
    handles.append(mpatches.Patch(facecolor="white", edgecolor="black", label="Unlabeled"))

    for ax, (clf, y_train, title) in zip(axes, classifiers):
        DecisionBoundaryDisplay.from_estimator(
            clf,
            X,
            response_method="predict_proba",
            plot_method="contourf",
            ax=ax,
            multiclass_colors=region_colors,
        )
        colors = [label_color_map[label] for label in y_train]
        ax.scatter(X[:, 0], X[:, 1], c=colors, edgecolor="black")
        ax.set_title(title)
    fig.suptitle(f"Semi-supervised decision boundaries (HyperParams: {hyper_param_ls} & {hyper_param_svc}", y=1)
    fig.legend(handles=handles, loc="lower center", ncol=len(handles), bbox_to_anchor=(0.5, 0.0))
    plt.show()

In [ ]:
# Final predictions
from sklearn.svm import SVC

X = df[['Sensor 2', 'Sensor 9', 'Sensor 13']].values
X = np.concat([X, np.sqrt(np.sum(X * X, axis=1)).reshape(-1, 1)], axis=1)
X = X[:, [1, 3]]  # Sensor 9, R
df["label_label_spreading"] = df["Label"].fillna(-1)
y = df["label_label_spreading"].values

labels = df['label_label_spreading'].values
unique_labels = sorted(set(labels))

svc_cls = SVC(probability=True, random_state=42, gamma=0.7)
# Self-trained classifier
ssl_svc_cls = SelfTrainingClassifier(svc_cls).fit(X, y)
y_hat = ssl_svc_cls.predict_proba(X)
df = pd.concat([
    df,
    pd.DataFrame(y_hat, columns=[f"Probability_label_{cls_label}" for cls_label in ssl_svc_cls.classes_]),
    # Labels are sorted so can just do +1 to get the class
    pd.DataFrame(1+y_hat.argmax(axis=1), columns=["Predicted_Label"]),
],axis=1)

df.drop(columns=["label_label_spreading"], inplace=True)

In [ ]:
assert len(df[(df["Label"]!=df["Predicted_Label"]) & (~df["Label"].isna())])==0, "Predicted label is not equal to the annotated label"